# CEE / ENV 586
## Fall 2023

In [ ]:
import hydrodata.point_observations.pandas.collect_observations as point_obs
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np

### Example workflow: 
#### Request streamflow data for a single site, for a single Water Year. Then plot the time series.

In [ ]:
# Collect streamflow data for a single site for a single Water Year
site_id = '01401000'
site_df, metadata_df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                              temporal_resolution='daily', aggregation='average', 
                                              date_start='1990-10-01', date_end='1991-09-30',
                                              site_ids=[site_id], return_metadata=True)

# Extract site name from the metadata (for plot title)
site_name = metadata_df['site_name'][metadata_df['site_id'] == site_id][0]
site_name

# Transpose and format observations dates
df_plot = site_df.drop(columns=['site_id', 'num_obs']).T
df_plot.index = pd.to_datetime(df_plot.index)

# Plot
plt.plot(df_plot)
plt.xlabel('Date')
plt.ylabel('Streamflow (m^3/s)')
plt.title(f'Streamflow for site {site_id} for Water Year 1991\n{site_name}')

site_names = metadata_df['site_name']
plt.legend(site_names)

plt.show()

# If you want to save a version, uncomment the following with your desired file path
# plt.savefig(f'path/to/save/streamflow_{site_id}.png')

### Example workflow: 
#### Request water table depth data for a single site, for a single Water Year. Then plot the time series.

In [ ]:
# Collect streamflow data for a single site for a single Water Year
site_id = '402032074392501'
site_df, metadata_df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='wtd', 
                                              temporal_resolution='daily', aggregation='average', 
                                              date_start='1990-10-01', date_end='1991-09-30',
                                              site_ids=[site_id], return_metadata=True)

# Extract site name from the metadata (for plot title)
site_name = metadata_df['site_name'][metadata_df['site_id'] == site_id][0]
site_name

# Transpose and format observations dates
df_plot = site_df.drop(columns=['site_id', 'num_obs']).T
df_plot.index = pd.to_datetime(df_plot.index)

# Plot
plt.plot(df_plot)
plt.xlabel('Date')
plt.ylabel('Water Table Depth (m)')
plt.title(f'Water Table Depth for site {site_id} for Water Year 1991\n{site_name}')

site_names = metadata_df['site_name']
plt.legend(site_names)

plt.show()

# If you want to save a version, uncomment the following with your desired file path
# plt.savefig(f'path/to/save/wtd_{site_id}.png')

### Example workflow: 
#### Request streamflow data for all sites within a bounding box, for a single Water Year. Then plot the time series all together.

In [ ]:
# Collect streamflow data for a single Water Year, for all sites within a latitude/longitude range.
latitude_range = (39, 40)
longitude_range = (-75, -74)

site_df, metadata_df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                              temporal_resolution='daily', aggregation='average', 
                                              date_start='1990-10-01', date_end='1991-09-30',
                                              latitude_range = latitude_range, longitude_range = longitude_range, 
                                              return_metadata=True)

# Transpose and format observations dates
df_plot = site_df.drop(columns=['site_id', 'num_obs']).T
df_plot.index = pd.to_datetime(df_plot.index)

# Plot
plt.plot(df_plot)
plt.xlabel('Date')
plt.ylabel('Streamflow (m^3/s)')
plt.title(f'Streamflow for for Water Year 1991')

# Uncomment these lines for a legend of corresponding site names
# site_names = metadata_df['site_name']
# plt.legend(site_names)

plt.show()

# If you want to save a version, uncomment the following with your desired file path
# plt.savefig(f'path/to/save/streamflow.png')

### Example workflow: 
#### Map the requested bounding box and overlay locations of points with data

In [ ]:
# First let's define a function to convert lon/lat to mercator coordinates for Bokeh mapping procedure
def wgs84_to_web_mercator(lon, lat):

    k = 6378137
    x = lon * (k * np.pi/180.0)
    y = np.log(np.tan((90 + lat) * np.pi/360.0)) * k

    return x, y

In [ ]:
# Collect streamflow data for a single Water Year, for all sites within a latitude/longitude range.
latitude_range = (39, 40)
longitude_range = (-75, -74)

site_df, metadata_df = point_obs.get_pandas_observations(data_source='usgs_nwis', variable='streamflow', 
                                              temporal_resolution='daily', aggregation='average', 
                                              date_start='1990-10-01', date_end='1991-09-30',
                                              latitude_range = latitude_range, longitude_range = longitude_range, 
                                              return_metadata=True)

In [ ]:
# Calculate and add mercator coordinates to metadata DataFrame
metadata_df['x'] = metadata_df.apply(lambda x: wgs84_to_web_mercator(x['longitude'], x['latitude'])[0], axis=1)
metadata_df['y'] = metadata_df.apply(lambda x: wgs84_to_web_mercator(x['longitude'], x['latitude'])[1], axis=1)

In [ ]:
# Plot
from bokeh.plotting import figure, output_notebook, show
from bokeh.tile_providers import OSM, get_provider
from bokeh.models import ColumnDataSource

output_notebook()
tile_provider = get_provider(OSM)

# Range bounds supplied in web mercator coordinates
# Translate min and max bounding values from metadata
p = figure(x_range=(wgs84_to_web_mercator(longitude_range[1], latitude_range[1])[0], 
                    wgs84_to_web_mercator(longitude_range[0], latitude_range[0])[0]),
           y_range=(wgs84_to_web_mercator(longitude_range[0], latitude_range[0])[1], 
                    wgs84_to_web_mercator(longitude_range[1], latitude_range[1])[1]),
           x_axis_type="mercator", y_axis_type="mercator")
p.add_tile(tile_provider)

# Define data source for point overlays
source = ColumnDataSource(
    data=dict(x=list(metadata_df['x']),
              y=list(metadata_df['y']))
)

# Overlay red circles onto map for site locations
p.circle(x="x", y="y", size=15, fill_color="red", fill_alpha=0.8, source=source)

show(p)